In [ ]:
import io
import time
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from skimage.feature import hog
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
dataset_file = hf_hub_download(
    repo_id='bazyl/GTSRB',
    filename='data/test-00000-of-00001-747a54d4a6461a97.parquet',
    repo_type='dataset'
)
table = pq.read_table(dataset_file)
total_samples = table.num_rows

In [ ]:
def extract_features(parquet_table, target_size=(32, 32)):
    X_list = []
    y_list = []
    for i in range(parquet_table.num_rows):
        cid = parquet_table['ClassId'][i].as_py()
        raw_b = parquet_table['Path'][i].as_py()['bytes']
        img = Image.open(io.BytesIO(raw_b)).convert('L').resize(target_size)
        arr = np.array(img, dtype=np.float32) / 255.0
        feat = hog(
            arr,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm='L2-Hys'
        )
        X_list.append(feat)
        y_list.append(cid)
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.int64)

In [ ]:
X, y = extract_features(table)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
models = {
    'Linear SVM': LinearSVC(C=1.0, dual='auto', max_iter=3000, random_state=42),
    'k-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
    'Random Forest': RandomForestClassifier(n_estimators=150, max_depth=20, random_state=42, n_jobs=-1)
}

trained_models = {}
eval_results = {}

for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    t_train = time.time() - t0
    t1 = time.time()
    y_pred = model.predict(X_test)
    t_pred = time.time() - t1
    acc = accuracy_score(y_test, y_pred)
    rep = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    trained_models[name] = model
    eval_results[name] = {
        'Accuracy': acc,
        'Macro Precision': rep['macro avg']['precision'],
        'Macro Recall': rep['macro avg']['recall'],
        'Macro F1': rep['macro avg']['f1-score'],
        'Train Time (s)': t_train,
        'Inference Time (s)': t_pred,
        'Predictions': y_pred
    }

In [ ]:
metrics_df = pd.DataFrame([
    {
        'Model': k,
        'Accuracy': v['Accuracy'],
        'Macro Precision': v['Macro Precision'],
        'Macro Recall': v['Macro Recall'],
        'Macro F1': v['Macro F1'],
        'Train Time (s)': v['Train Time (s)'],
        'Inference Time (s)': v['Inference Time (s)']
    }
    for k, v in eval_results.items()
])
print(metrics_df.to_string(index=False))

In [ ]:
svm_preds = eval_results['Linear SVM']['Predictions']
correct_idx = np.where(svm_preds == y_test)[0][0]
wrong_idx = np.where(svm_preds != y_test)[0][1]

print(f'Correct Case -> Test Index: {correct_idx}, Ground Truth: {y_test[correct_idx]}, Predicted: {svm_preds[correct_idx]}')
print(f'Incorrect Case -> Test Index: {wrong_idx}, Ground Truth: {y_test[wrong_idx]}, Predicted: {svm_preds[wrong_idx]}')

In [ ]:
cm = confusion_matrix(y_test, svm_preds)
print(f'Confusion matrix shape: {cm.shape}')
print(f'Total misclassified instances: {np.sum(svm_preds != y_test)} out of {len(y_test)}')